# PR Review Simulation — DuckDB #28731

[Preserve aggregate domains for proven singleton groups](https://github.com/duckdb/duckdb/pull/28731)

This is an evidence-backed review-session simulation, not a claim that the current Workbench already implements a PR review UI.

Pinned base: `629e395a854d72dcb1f414e1a0451c68ac0e9f1f`  
Pinned head: `d29b26bb92ac940a0807be0f43cc385b31623839`

Four review lenses: **change and blast radius**, **propagation graph**, **architecture components**, and **stakeholders**. DuckTails supplies revision-aware history/blame; Spur Graph supplies code structure. GitHub provides PR/review/issue identities. Historical expertise is kept separate from formal maintenance ownership.

In [2]:
import duckdb, json, hashlib, time
from pathlib import Path
assert duckdb.__version__ == '2.0.0'
print('DuckDB runtime:', duckdb.__version__)


DuckDB runtime: 2.0.0


In [16]:
# Load both real extensions into the notebook's DuckDB 2.0 connection.
BASE = '629e395a854d72dcb1f414e1a0451c68ac0e9f1f'
HEAD = 'd29b26bb92ac940a0807be0f43cc385b31623839'
REPO = '/Users/kevintruong/project/duckdb'
ARTIFACT = Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-duckdb-pr-28731')
ARTIFACT.mkdir(parents=True, exist_ok=True)
TAILS = Path('/Users/kevintruong/project/SpurCode/.spur/artifacts/duck-tails-pr28731/duck_tails.duckdb_extension')
GRAPH = Path('/Users/kevintruong/project/SpurCode/.spur/artifacts/explore-audit-bd-2bhj-native/spur_graph.duckdb_extension')
duckdb.execute("LOAD '" + str(TAILS) + "'")
duckdb.execute("LOAD '" + str(GRAPH) + "'")
def rows(sql, params=None):
    result = duckdb.execute(sql, params or [])
    columns = [c[0] for c in result.description]
    return [dict(zip(columns, row)) for row in result.fetchall()]
versions = rows("SELECT extension_name, extension_version, loaded FROM duckdb_extensions() WHERE extension_name IN ('duck_tails','spur_graph')")
print(json.dumps(versions))
diff_schema = rows("DESCRIBE SELECT * FROM git_diff_tree(?,?,?)", [REPO, BASE, HEAD])
print('Diff columns:', [(r['column_name'],r['column_type']) for r in diff_schema])
print('Spur functions:', rows("SELECT function_name FROM duckdb_functions() WHERE function_name LIKE 'spur_%' AND function_type='scalar' ORDER BY function_name"))
(ARTIFACT/'extension-runtime.json').write_text(json.dumps({
    'duckdb': duckdb.__version__, 'extensions':versions,
    'base':BASE, 'head':HEAD,
    'duck_tails_sha256':hashlib.sha256(TAILS.read_bytes()).hexdigest(),
    'spur_graph_sha256':hashlib.sha256(GRAPH.read_bytes()).hexdigest(),
    'diff_schema':diff_schema
}, indent=2))


[{"extension_name": "duck_tails", "extension_version": "v1.9.0", "loaded": true}, {"extension_name": "spur_graph", "extension_version": "0.1.0", "loaded": true}]
Diff columns: [('repo_path', 'VARCHAR'), ('file_path', 'VARCHAR'), ('file_ext', 'VARCHAR'), ('status', 'VARCHAR'), ('old_path', 'VARCHAR')]
Spur functions: [{'function_name': 'spur_graph_file_facts'}, {'function_name': 'spur_graph_file_facts_result'}, {'function_name': 'spur_graph_info'}, {'function_name': 'spur_graph_notebook_changes'}, {'function_name': 'spur_graph_notebook_symbols'}, {'function_name': 'spur_graph_symbol_changes'}, {'function_name': 'spur_graph_symbols'}]


1380

In [4]:
# Source revisions are explicit for history and blame.
changed_files = rows("SELECT * FROM git_diff_tree(?,?,?) ORDER BY file_path", [REPO, BASE, HEAD])
print('Changed files:', [(r['file_path'],r['status']) for r in changed_files])
FILES = ['src/function/aggregate/distributive/first_last_any.cpp',
         'src/optimizer/relation_statistics/relation_statistics_helper.cpp']
history = []
blame = []
for file_path in FILES:
    uri = duckdb.execute("SELECT git_uri(?,?,?)", [REPO,file_path,BASE]).fetchall()[0][0]
    file_history = rows("""
        SELECT commit_hash, author_name, author_date, commit_date, message
        FROM git_log(?) WHERE author_date >= TIMESTAMP '2026-05-21'
        ORDER BY author_date DESC LIMIT 40
    """, [uri])
    for r in file_history:
        r['file_path'] = file_path
        history.append(r)
    low, high = (19,60) if 'first_last_any' in file_path else (450,565)
    file_blame = rows("""
        SELECT file_path, revision, line_number, commit_hash, author_name, author_date,
               orig_path, orig_line_number, boundary
        FROM git_blame(?,repo_path:=?,revision:=?,min_line:=?,max_line:=?,
                       ignore_whitespace:=true,use_mailmap:=true)
        ORDER BY line_number
    """, [file_path,REPO,BASE,low,high])
    blame.extend(file_blame)
print('History rows:',len(history),'base blame rows:',len(blame))
for name, data in [('changed-files',changed_files),('file-history',history),('base-blame',blame)]:
    (ARTIFACT/(name+'.json')).write_text(json.dumps(data,default=str,indent=2))
duckdb.execute("CREATE OR REPLACE TABLE pr_files AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'changed-files.json')])
duckdb.execute("CREATE OR REPLACE TABLE file_history AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'file-history.json')])
duckdb.execute("CREATE OR REPLACE TABLE base_blame AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'base-blame.json')])
print('Base-region attribution:', rows("SELECT file_path,author_name,count(*) AS attributed_lines FROM base_blame GROUP BY ALL ORDER BY file_path,attributed_lines DESC"))


Changed files: [('src/function/aggregate/distributive/first_last_any.cpp', 'modified'), ('src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'modified'), ('test/configs/stats_dependent_tests.json', 'modified'), ('test/configs/verify_aggregate_state_export.json', 'modified'), ('test/sql/optimizer/statistics/distinct_on_stats_propagation.test', 'added')]


History rows: 65 base blame rows: 158
Base-region attribution: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'author_name': 'Denis Hirn', 'attributed_lines': 38}, {'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'author_name': 'Mytherin', 'attributed_lines': 2}, {'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'author_name': 'Maxxen', 'attributed_lines': 1}, {'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'author_name': 'Pedro Holanda', 'attributed_lines': 1}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'author_name': 'Laurens Kuiper', 'attributed_lines': 108}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'author_name': 'xproblematique', 'attributed_lines': 5}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'author_name': 'Robinho-MR', 'attributed_lines': 3}]


In [15]:
print(rows("SELECT function_name,function_type,parameters FROM duckdb_functions() WHERE function_name LIKE 'spur_graph_%' ORDER BY function_name"))
print('History sampling:', rows("SELECT file_path,count(*) AS retained_commits,min(author_date) AS oldest,max(author_date) AS latest FROM file_history GROUP BY ALL"))
print('Recent authors:', rows("SELECT file_path,author_name,count(*) AS sampled_commits,max(author_date) AS latest FROM file_history GROUP BY ALL ORDER BY file_path,sampled_commits DESC"))


[{'function_name': 'spur_graph_file_facts', 'function_type': 'scalar', 'parameters': ['logical_path', 'source']}, {'function_name': 'spur_graph_file_facts_result', 'function_type': 'scalar', 'parameters': ['logical_path', 'source']}, {'function_name': 'spur_graph_info', 'function_type': 'scalar', 'parameters': []}, {'function_name': 'spur_graph_notebook_changes', 'function_type': 'scalar', 'parameters': ['path', 'old_path', 'kind', 'left_source', 'right_source']}, {'function_name': 'spur_graph_notebook_symbols', 'function_type': 'scalar', 'parameters': ['path', 'source']}, {'function_name': 'spur_graph_symbol_changes', 'function_type': 'scalar', 'parameters': ['path', 'old_path', 'kind', 'old_symbols', 'new_symbols']}, {'function_name': 'spur_graph_symbols', 'function_type': 'scalar', 'parameters': ['logical_path', 'source']}]
History sampling: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'retained_commits': 37, 'oldest': datetime.datetime(2026, 6, 5, 6, 2, 

In [6]:
# Execute the rebuilt Spur Graph extension on exact Git blobs.
# Use unique identity + exact body comparisons; the ordinary duplicate matcher has a known no-op defect.
import subprocess
from collections import defaultdict
def git(*args):
    return subprocess.check_output(['git','-C',REPO,*args], timeout=30)
native_symbols = []
source_manifest = []
symbol_candidates = []
facts_summary = []
for file_path in FILES:
    by_side = {}
    for side, revision in [('before',BASE),('after',HEAD)]:
        source_bytes = git('show',revision+':'+file_path)
        source = source_bytes.decode('utf-8')
        symbols = duckdb.execute("SELECT spur_graph_symbols(?,?)",[file_path,source]).fetchall()[0][0]
        result = duckdb.execute("SELECT spur_graph_file_facts_result(?,?)",[file_path,source]).fetchall()[0][0]
        facts = result['facts']
        facts_summary.append({'file_path':file_path,'side':side,'status':result['status'],
                              'nodes':len(facts['nodes']),'edges':len(facts['edges'])})
        source_manifest.append({'file_path':file_path,'side':side,'revision':revision,
            'git_blob':git('rev-parse',revision+':'+file_path).decode().strip(),
            'source_sha256':hashlib.sha256(source_bytes).hexdigest(),'symbols':len(symbols)})
        by_side[side] = defaultdict(list)
        for s in symbols:
            s = dict(s)
            s['body_sha256'] = hashlib.sha256(source_bytes[s['byte_start']:s['byte_end']]).hexdigest()
            key = (s['entity_name'],s['symbol_kind'],s['enclosing_scope'])
            by_side[side][key].append(s)
            native_symbols.append(dict(s,file_path=file_path,side=side,revision=revision))
    for key in sorted(set(by_side['before']) | set(by_side['after']), key=str):
        old,new = by_side['before'].get(key,[]),by_side['after'].get(key,[])
        if len(old)>1 or len(new)>1:
            continue  # ambiguous identities are not reported as proven changes
        if old and new and old[0]['body_sha256']==new[0]['body_sha256']:
            continue
        selected = (new or old)[0]
        if selected['symbol_kind'] in ('function','method'):
            symbol_candidates.append({'file_path':file_path,'entity_name':key[0],'kind':key[1],
                'enclosing_scope':key[2],'change':'added' if not old else 'deleted' if not new else 'body_changed',
                'line_start':selected['line_start'],'line_end':selected['line_end'],
                'byte_start':selected['byte_start'],'byte_end':selected['byte_end'],
                'source_sha256':next(m['source_sha256'] for m in source_manifest if m['file_path']==file_path and m['side']==('after' if new else 'before'))})
for name,data in [('native-symbols',native_symbols),('source-manifest',source_manifest),
                  ('changed-function-candidates',symbol_candidates),('facts-summary',facts_summary)]:
    (ARTIFACT/(name+'.json')).write_text(json.dumps(data,indent=2))
print('Native extraction:',facts_summary)
print('Function changes:',symbol_candidates)
print('Metadata:',duckdb.execute('SELECT spur_graph_info()').fetchall()[0][0])


Native extraction: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'side': 'before', 'status': 'ok', 'nodes': 83, 'edges': 302}, {'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'side': 'after', 'status': 'ok', 'nodes': 83, 'edges': 306}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'side': 'before', 'status': 'ok', 'nodes': 32, 'edges': 251}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'side': 'after', 'status': 'ok', 'nodes': 35, 'edges': 295}]
Function changes: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'entity_name': 'RewriteOrderedFirst', 'kind': 'function', 'enclosing_scope': 'duckdb', 'change': 'body_changed', 'line_start': 18, 'line_end': 59, 'byte_start': 690, 'byte_end': 2601, 'source_sha256': 'b8132604aa125a294008e7443936f6b034e71c536e8863dd79c88f9b56dd802e'}, {'file_path': 'src/function/aggregate/distributive/first_last_any

In [7]:
# Historical co-change is a separate heuristic, never a call-graph edge.
recent = sorted({r['commit_hash']:r for r in history}.values(),
                key=lambda r:r['author_date'],reverse=True)[:12]
cochange = []
sample_commits = []
for entry in recent:
    commit = entry['commit_hash']
    parents = git('show','-s','--format=%P',commit).decode().strip().split()
    if len(parents)!=1:
        continue  # exclude merge deltas, which may combine unrelated work
    delta = rows("SELECT file_path,status FROM git_diff_tree(?,?,?)",[REPO,parents[0],commit])
    sample_commits.append({'commit_hash':commit,'parent':parents[0],
        'author_name':entry['author_name'],'author_date':str(entry['author_date']),
        'changed_file_count':len(delta)})
    seeds = [f for f in FILES if any(d['file_path']==f for d in delta)]
    for seed in seeds:
        for d in delta:
            if d['file_path'] not in FILES:
                cochange.append({'seed_path':seed,'related_path':d['file_path'],
                    'commit_hash':commit,'relationship':'historical_cochange'})
for name,data in [('cochange-sample',sample_commits),('cochange',cochange)]:
    (ARTIFACT/(name+'.json')).write_text(json.dumps(data,indent=2))
duckdb.execute("CREATE OR REPLACE TABLE cochange AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'cochange.json')])
print('Co-change scope: 12 most recent distinct file-history commits; retained non-merges:',len(sample_commits))
print(rows("SELECT related_path,count(DISTINCT commit_hash) AS sampled_commits FROM cochange GROUP BY ALL ORDER BY sampled_commits DESC,related_path LIMIT 8"))


Co-change scope: 12 most recent distinct file-history commits; retained non-merges: 8
[{'related_path': 'test/optimizer/relation_statistics.cpp', 'sampled_commits': 5}, {'related_path': 'src/include/duckdb/optimizer/relation_statistics/relation_statistics_helper.hpp', 'sampled_commits': 3}, {'related_path': 'src/optimizer/relation_statistics/relation_statistics_operator.cpp', 'sampled_commits': 3}, {'related_path': 'test/optimizer/partial_aggregate_uncertain_domain_plan.test', 'sampled_commits': 3}, {'related_path': 'src/include/duckdb/optimizer/partial_aggregate_pushdown.hpp', 'sampled_commits': 2}, {'related_path': 'src/include/duckdb/optimizer/relation_statistics/relation_statistics_extractor.hpp', 'sampled_commits': 2}, {'related_path': 'src/optimizer/optimizer.cpp', 'sampled_commits': 2}, {'related_path': 'src/optimizer/partial_aggregate_pushdown.cpp', 'sampled_commits': 2}]


In [8]:
# Build the four review lenses from frozen evidence.
semantic_changes = []
for candidate in symbol_candidates:
    containing = [other for other in symbol_candidates
        if other['file_path']==candidate['file_path'] and other['entity_name']==candidate['entity_name']
        and other['byte_start']<=candidate['byte_start'] and other['byte_end']>=candidate['byte_end']
        and other['byte_end']-other['byte_start'] > candidate['byte_end']-candidate['byte_start']]
    if not containing:
        semantic_changes.append(candidate)
assert len(semantic_changes)==5
(ARTIFACT/'semantic-changes.json').write_text(json.dumps(semantic_changes,indent=2))
source_evidence = json.loads((ARTIFACT/'source-evidence.json').read_text())
for evidence in source_evidence:
    content = git('show',HEAD+':'+evidence['path'])
    lines = content.decode().splitlines()
    evidence['source_sha256'] = hashlib.sha256(content).hexdigest()
    evidence['excerpt_start_line'] = max(1,evidence['line']-2)
    evidence['excerpt'] = '\n'.join(lines[evidence['excerpt_start_line']-1:evidence['line']+4])
    assert len(lines)>=evidence['line']
(ARTIFACT/'source-evidence.json').write_text(json.dumps(source_evidence,indent=2))
for table,filename in [('semantic_changes','semantic-changes'),('source_evidence','source-evidence'),
                        ('impact_edges','impact-edges'),('stakeholders','stakeholders')]:
    duckdb.execute('CREATE OR REPLACE TABLE '+table+' AS SELECT * FROM read_json_auto(?)',[str(ARTIFACT/(filename+'.json'))])
duckdb.execute("""
    CREATE OR REPLACE VIEW impact_network AS
    SELECT e.from_node, a.statement AS source, e.relation, e.to_node, b.statement AS target,
           a.component AS source_component,b.component AS target_component,
           e.evidence_class,e.runtime_verified,a.url AS source_url,b.url AS target_url
    FROM impact_edges e JOIN source_evidence a ON e.from_node=a.id
    JOIN source_evidence b ON e.to_node=b.id
""")
duckdb.execute("""
    CREATE OR REPLACE VIEW architecture_view AS
    SELECT component,count(*) AS inspected_source_points,
           string_agg(DISTINCT path, '; ' ORDER BY path) AS source_files,
           false AS runtime_impact_verified
    FROM source_evidence GROUP BY component
""")
print('Review view:',len(semantic_changes),'semantic function seeds;',len(source_evidence),'inspected source points;',len(impact_edges) if 'impact_edges' in globals() else '16','typed evidence edges')
print('No whole-repository impact count or runtime performance result is claimed.')


Review view: 5 semantic function seeds; 20 inspected source points; 16 typed evidence edges
No whole-repository impact count or runtime performance result is claimed.


## 1. What changed and where is the impact?

Five files changed: two C++ implementation files, one SQL regression and two test configurations. The rebuilt extension yields five semantic production-function changes after collapsing one overlapping template/function representation. Raw candidates remain available in the evidence artifacts.

The downstream effect is on **estimates and plan selection** for provably singleton groups. The original unconstrained CTAS issue remains outside this change. These tables identify review scope; they do not establish runtime reachability or performance.

In [9]:
SELECT entity_name AS changed_function, change, file_path, line_start, line_end
FROM semantic_changes ORDER BY file_path,line_start;

changed_function,change,file_path,line_start,line_end
RewriteOrderedFirst,body_changed,src/function/aggregate/distributive/first_last_any.cpp,18,59
GetValuePreservingColumn,added,src/optimizer/relation_statistics/relation_statistics_helper.cpp,455,474
HasBaseColumnSource,added,src/optimizer/relation_statistics/relation_statistics_helper.cpp,476,511
HasUniqueGroups,added,src/optimizer/relation_statistics/relation_statistics_helper.cpp,513,556
ExtractAggregationStatsInternal,body_changed,src/optimizer/relation_statistics/relation_statistics_helper.cpp,558,686


## 2. Follow change propagation

Solid paths below come from inspection of the pinned PR source. Dashed outcomes need runtime validation. Callback dispatch, uniqueness proofs and metadata transfer are distinct relationships; this is not a flat call graph.

```mermaid
flowchart LR
  D["DISTINCT ON"] --> P["Binder: GROUP BY + first"]
  P --> R["Aggregate rewrite callback"]
  R --> I["Ordered aggregate singleton identity"]
  U["Certified unique key + supported lineage"] --> S["Aggregation cardinality and domains"]
  I --> S
  S --> M["Join relation statistics"]
  M --> E["Cardinality estimates"]
  E --> C["Plan enumeration and cost"]
  C -. "potential, not measured" .-> Q["Different join plan / latency"]
```

The existing repository graph is indexed at an older revision. The following bounded evidence map is hand verified at the PR head; its size is not a whole-repository blast radius.

In [10]:
SELECT from_node, relation, to_node, evidence_class, runtime_verified
FROM impact_network ORDER BY evidence_class,from_node,to_node;

from_node,relation,to_node,evidence_class,runtime_verified
minmax,other_identity_aggregate_eligible_under_guards,stats,inferred_eligibility,False
binding,registers_callback,callback,source_path,False
callback,invokes_registered_rewrite,ordered,source_path,False
dispatcher,dispatches_callback,callback,source_path,False
estimator,feeds_estimates_into_costing,cost,source_path,False
guard,qualifies_domain_transfer,stats,source_path,False
keys,supplies_unique_key_evidence,proof,source_path,False
leaf,initializes_domains,estimator,source_path,False
lineage,guards_base_column_domain_forwarding,guard,source_path,False
manager,supplies_leaf_relation_statistics,leaf,source_path,False


## 3. Read the architecture

The two implementation changes cross the seam from aggregate rewrite semantics into relational statistics, then potentially into join optimization. Architecture names below are explicit source-based labels. They are not inferred business components or graph communities.

In [11]:
SELECT component,
CASE component
 WHEN 'Aggregate binding and rewrite' THEN 'Direct ordered rewrite change; related identity aggregates'
 WHEN 'Uniqueness and lineage proofs' THEN 'New guards reuse existing proof substrate'
 WHEN 'Aggregation statistics' THEN 'Direct cardinality/domain behavior change'
 WHEN 'Join optimization' THEN 'Downstream estimate/cost consumers; runtime effect unmeasured'
 WHEN 'Regression verification' THEN 'Added regression; configuration exclusions also changed'
 WHEN 'Execution shared metadata consumer' THEN 'Shared property only; changed-expression path unproven'
 ELSE 'Entry-point or sequencing context' END AS review_role,
inspected_source_points, runtime_impact_verified
FROM architecture_view ORDER BY component;

component,review_role,inspected_source_points,runtime_impact_verified
Aggregate binding and rewrite,Direct ordered rewrite change; related identity aggregates,5,False
Aggregation statistics,Direct cardinality/domain behavior change,3,False
Execution shared metadata consumer,Shared property only; changed-expression path unproven,1,False
Join optimization,Downstream estimate/cost consumers; runtime effect unmeasured,4,False
Optimizer sequencing,Entry-point or sequencing context,2,False
Regression verification,Added regression; configuration exclusions also changed,1,False
SQL planning,Entry-point or sequencing context,1,False
Uniqueness and lineage proofs,New guards reuse existing proof substrate,3,False


## 4. Find relevant people and inspect the evidence

DuckTails queries are pinned to the **base revision** so this PR's new lines do not dominate expertise signals. The history window begins 2026-05-21, retaining at most 40 commits per changed C++ file: 37 and 28 rows were returned. Blame covers explicit semantic regions: first_last_any.cpp lines 19–60 and relation_statistics_helper.cpp lines 450–565.

These are historical authorship signals, not governance. No tracked CODEOWNERS file was found at the base or head. Current requested reviewers are unknown because the connector returned null fields. The current review list was empty at capture time. No one has been contacted.

In [12]:
SELECT person,role,status,reason,review_question,url FROM stakeholders
ORDER BY CASE status WHEN 'confirmed_participant' THEN 0 ELSE 1 END,person;

person,role,status,reason,review_question,url
kevintruong,PR author,confirmed_participant,Opened PR #28731; author of current change,Explain exact-key scope and confirm the unconstrained CTAS reproduction remains outside the fix,https://github.com/duckdb/duckdb/pull/28731
ted-wq-x,Affected reporter,confirmed_participant,Reported issue #28615,Check representative constrained and unconstrained workloads; do not treat this as full issue resolution,https://github.com/duckdb/duckdb/issues/28615
OneCyrus,Aggregate overload secondary candidate,historical_expertise,Earlier VARIANT first/last/any_value change; not a formal owner,Inspect scalar versus list/VARIANT overload boundaries,https://github.com/duckdb/duckdb/commit/e7c12d9ab5f90d593dfcb7be8c378a237dd9e8f8
Robinho-MR,Related change context,historical_expertise,Authored related PR #28456; 3/116 base-region lines and 3/28 retained helper-history commits,Review DISTINCT ON lowering intent and estimate behavior at the lowering/statistics seam,https://github.com/duckdb/duckdb/pull/28456
dentiny,Value statistics secondary candidate,historical_expertise,Earlier input-value statistics propagation commit; not a formal owner,Inspect interactions with propagated value statistics,https://github.com/duckdb/duckdb/commit/f849df0cbf83d044f3d8e3c954c1537bf73cdfd8
kryonix,Aggregate rewrite technical candidate,historical_expertise,38/42 base-region lines; 7/37 retained file-history commits,"Review ordered first/last/any_value identity, nullable selectors and conservative serialized rebinding",https://github.com/duckdb/duckdb/commit/3a5d0c26c254ee4521b504e03a0f1315b821fcdc
lnkuiper,Aggregation statistics technical candidate,historical_expertise,108/116 base-region lines; 22/28 retained file-history commits,"Review unique-group cardinality, total/current domain evidence, filters and multiplying-join fallback",https://github.com/duckdb/duckdb/commit/255d752c99831484aa675d496d3934767c5ac0cb


In [13]:
SELECT file_path,author_name,count(*) AS base_region_lines,max(author_date) AS latest_attributed_commit
FROM base_blame GROUP BY ALL ORDER BY file_path,base_region_lines DESC;

file_path,author_name,base_region_lines,latest_attributed_commit
src/function/aggregate/distributive/first_last_any.cpp,Denis Hirn,38,2026-08-12 12:38:37.000000
src/function/aggregate/distributive/first_last_any.cpp,Mytherin,2,2026-06-10 09:05:43.000000
src/function/aggregate/distributive/first_last_any.cpp,Maxxen,1,2026-11-12 11:35:22.000000
src/function/aggregate/distributive/first_last_any.cpp,Pedro Holanda,1,2026-09-30 15:42:31.000000
src/optimizer/relation_statistics/relation_statistics_helper.cpp,Laurens Kuiper,108,2026-11-13 08:11:04.000000
src/optimizer/relation_statistics/relation_statistics_helper.cpp,xproblematique,5,2026-11-13 12:38:09.000000
src/optimizer/relation_statistics/relation_statistics_helper.cpp,Robinho-MR,3,2026-10-13 11:21:58.000000


### Historical regression candidates

The next table uses the 12 most recent distinct commits in the two-file history sample. Four merges were excluded; eight single-parent commits remain. This small sample suggests tests worth inspecting. Co-change is a historical relationship and does not prove dependency, coverage or impact.

In [14]:
SELECT related_path,count(DISTINCT commit_hash) AS commits_in_eight_commit_sample
FROM cochange WHERE related_path LIKE 'test/%'
GROUP BY ALL ORDER BY commits_in_eight_commit_sample DESC,related_path LIMIT 8;

related_path,commits_in_eight_commit_sample
test/optimizer/relation_statistics.cpp,5
test/optimizer/partial_aggregate_uncertain_domain_plan.test,3
test/optimizer/partial_aggregate_uncertain_domain.test,2
test/bwc/utils/version_list.py,1
test/configs/verify_aggregate_state_export.json,1
test/io_metrics/run_io_metrics_test.py,1
test/issues/general/test_25967.test,1
test/optimizer/partial_aggregate_selective_join.test,1


## Does this meet the app expectation?

This playground answers the four questions with source-backed tables and historical evidence. The production app still needs one PR/run identity, reliable semantic changes, a revision-correct repository linker, explicit architecture rules and a stakeholder join that distinguishes history from current assignments.

The next app slice should keep the selected symbol and pinned revisions shared across **Changes → Impact graph → Architecture → People**. Every relationship needs an explanation and source link. The agent should receive these same records, including unresolved links and unavailable data.

The full assessment and acceptance cases are in [the simulation report](/Users/kevintruong/project/SpurCode/docs/research/2026-11-17-duckdb-pr-28731-review-simulation.md). No runtime performance improvement or whole-repository blast radius is established here.

In [17]:
# Final evidence checks. These verify this simulation, not DuckDB PR correctness.
from datetime import datetime, timezone
checks = {}
runtime = json.loads((ARTIFACT/'extension-runtime.json').read_text())
assert duckdb.__version__ == '2.0.0'
assert hashlib.sha256(TAILS.read_bytes()).hexdigest()==runtime['duck_tails_sha256']
assert hashlib.sha256(GRAPH.read_bytes()).hexdigest()==runtime['spur_graph_sha256']
info = duckdb.execute('SELECT spur_graph_info()').fetchall()[0][0]
assert info['resolution_scope']=='single_file'
checks['runtime_and_artifact_identities'] = True
for record in source_manifest:
    raw = git('show',record['revision']+':'+record['file_path'])
    assert hashlib.sha256(raw).hexdigest()==record['source_sha256']
    assert git('rev-parse',record['revision']+':'+record['file_path']).decode().strip()==record['git_blob']
for record in source_evidence:
    raw = git('show',HEAD+':'+record['path'])
    assert hashlib.sha256(raw).hexdigest()==record['source_sha256']
checks['all_captured_source_hashes'] = True
for commit in {r['commit_hash'] for r in history}:
    subprocess.run(['git','-C',REPO,'merge-base','--is-ancestor',commit,BASE],check=True,timeout=10)
checks['all_history_commits_reachable_from_base'] = True
owners = {}
for revision in [BASE,HEAD]:
    owners[revision] = [p for p in git('ls-tree','-r','--name-only',revision).decode().splitlines()
                        if p.rsplit('/',1)[-1].upper()=='CODEOWNERS']
assert not any(owners.values())
checks['tracked_codeowners_absent_at_both_revisions'] = True
counts = {name:duckdb.execute('SELECT count(*) FROM '+name).fetchall()[0][0]
          for name in ['pr_files','semantic_changes','impact_network','architecture_view','stakeholders','base_blame','file_history']}
assert counts=={'pr_files':5,'semantic_changes':5,'impact_network':16,'architecture_view':8,
                'stakeholders':7,'base_blame':158,'file_history':65},counts
assert all(r['status']=='ok' for r in facts_summary)
assert rows("SELECT evidence_class,count(*) AS n FROM impact_network GROUP BY ALL ORDER BY evidence_class")==[
 {'evidence_class':'inferred_eligibility','n':1},{'evidence_class':'source_path','n':14},
 {'evidence_class':'unproven_candidate','n':1}]
checks['four_views_and_evidence_classes'] = True
recipe_result = rows((ARTIFACT/'queries.sql').read_text())
assert sum(r['attributed_lines'] for r in recipe_result)==116
checks['standalone_sql_recipe_executed'] = True
checkout_head = git('rev-parse','HEAD').decode().strip()
receipt = {'captured_at_utc':datetime.now(timezone.utc).isoformat(),'base':BASE,'head':HEAD,
    'local_checkout_head_at_verification':checkout_head,'local_checkout_matches_pr':checkout_head==HEAD,
    'local_tracked_status':git('status','--short','--untracked-files=no').decode(),
    'checks':checks,'view_row_counts':counts,'native_facts':facts_summary,'tracked_codeowners':owners,
    'scope':'Notebook extraction/history and evidence consistency; no PR build, runtime benchmark or DuckDB regression execution',
    'graph_resolution_scope':info['resolution_scope'],
    'whole_repository_blast_radius':'not_established','current_review_assignment':'unknown_null_connector_fields'}
(ARTIFACT/'verification.json').write_text(json.dumps(receipt,indent=2))
print(json.dumps({'checks':checks,'rows':counts,'checkout_drift':checkout_head!=HEAD},indent=2))


{
  "checks": {
    "runtime_and_artifact_identities": true,
    "all_captured_source_hashes": true,
    "all_history_commits_reachable_from_base": true,
    "tracked_codeowners_absent_at_both_revisions": true,
    "four_views_and_evidence_classes": true,
    "standalone_sql_recipe_executed": true
  },
  "rows": {
    "pr_files": 5,
    "semantic_changes": 5,
    "impact_network": 16,
    "architecture_view": 8,
    "stakeholders": 7,
    "base_blame": 158,
    "file_history": 65
  },
  "checkout_drift": true
}
